**CI twin of `capstone.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv

bank = load_csv("bank-marketing-sample")
print(bank.shape)
print(bank["y"].value_counts(normalize=True).round(3))
bank.head(3)

In [ ]:
from sklearn.model_selection import train_test_split

y = (bank["y"] == "yes").astype(int)
X = bank.drop(columns=["y"])

train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"train {len(train_X)}, sealed test {len(test_X)} "
      f"({int(test_y.sum())} subscribers)")

In [ ]:
from lib.grader import between
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Steps 1-2 happened above. Step 6's landmine rule: duration is out.
feats = [c for c in train_X.columns if c != "duration"]
num = train_X[feats].select_dtypes(include="number").columns.tolist()
cat = [c for c in feats if c not in num]

prep = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

# One of several passing builds: the balanced logistic pipeline.
# (Our comparison run: logistic 0.728±0.026, forest 0.735±0.047,
#  boosted 0.733±0.043 CV AUC — a three-way tie inside the spreads;
#  we take the interpretable one, per Ch20.)
final_model = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
]).fit(train_X[feats], train_y)

sealed_auc = roc_auc_score(
    test_y, final_model.predict_proba(test_X[final_model.feature_names_in_])[:, 1])

run_tests([
    ("sealed set intact (the tripwire)",
     (len(test_X), int(test_X.index.to_numpy().sum())), (375, 286013)),
    ("train and test disjoint",
     len(set(train_X.index) & set(test_X.index)), 0),
    ("the landmine is not in the model's diet",
     "duration" in list(final_model.feature_names_in_), False),
    between("held-out ROC-AUC (grader-computed)", float(sealed_auc), 0.55, 1.0),
])